# TDA Comparison — Raw vs Imputed · Amsterdam vs Eindhoven

## Objective

This notebook quantifies how **LiDAR data imputation affects topological structure** in two Dutch urban terrain datasets:
- **Amsterdam Centre** — historic canal city, 39.6% original NoData (open water + building gaps)
- **Eindhoven North** — peri-urban, 24.1% original NoData (buildings + road gaps)

For each city we compare **four versions** of the same 1 × 1 km zoom AOI:

| | Raw (`.tif`) | Imputed (`.npy`) |
|---|---|---|
| **Amsterdam** | AHN DTM, NoData = open water / rooftop holes | IDW reconstruction after canal flood-fill |
| **Eindhoven** | AHN DTM, NoData = buildings / vegetation gaps | IDW reconstruction (Step 2 skipped — no dominant water body) |

---

## TDA Background

We use **sublevel-set persistent homology** via `gudhi.CubicalComplex`. The filtration $\{x : f(x) \le t\}$ is equivalent to raising a water level $t$ over the terrain $f$.

Each **$H_0$ persistence pair** $(b, d)$ corresponds to one basin:
- **Birth** $b$ = elevation of the basin's deepest point (local minimum — where water first pools)
- **Death** $d$ = elevation of the saddle where the basin overflows into a neighbour
- **Persistence** $p = d - b$ = **spillover depth** in metres — how deep the basin must fill before merging

High-persistence pairs are topologically robust (real flood-risk depressions); low-persistence pairs are noise (micro-roughness, sensor artefacts).

**What imputation changes topologically:**  
Filling NoData pixels introduces new terrain values at previously undefined locations. This can:
1. *Merge* previously isolated basins (if a gap was acting as an artificial barrier) → fewer high-persistence pairs
2. *Create* shallow new basins at imputed fill boundaries → more low-persistence noise
3. Leave the diagram *unchanged* if NoData was sparse and the IDW fill matches surrounding terrain well

We measure the overall shift in the persistence diagram using the **bottleneck distance** $d_B$, which is the minimax cost of the optimal matching between the two diagrams.

---

## Notebook Structure

| Cell group | What it does |
|---|---|
| **1 — Imports + paths** | Load libraries, set data paths and AOI coordinates |
| **2 — Data extraction** | Crop raw `.tif` and imputed `.npy` to the same zoom window; visualise 2 × 2 heatmaps |
| **3 — TDA** | Run sublevel-set persistence on all 4 crops; plot 2 × 2 persistence diagrams; save `.npz` |
| **4 — Comparison** | Bottleneck distances, sorted-persistence curves, summary table |

In [ ]:
# Cell 1 — imports
from pathlib import Path
import sys
import warnings
import gc

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import rasterio
from rasterio.windows import from_bounds as window_from_bounds

import gudhi as gd

print(f"numpy    {np.__version__}")
print(f"rasterio {rasterio.__version__}")
print(f"gudhi    {gd.__version__}")

In [ ]:
# Cell 2 — paths, AOIs, and water-level constants
#
# Data convention: raw .tif tiles live in <repo_root>/data/,
# imputed arrays in <repo_root>/output/ and <repo_root>/imputed_datasets/.
# We walk up the tree until we find requirements.txt to locate the repo root
# regardless of which directory the kernel was launched from.

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "requirements.txt not found. Make sure the kernel is running inside the repo."
    )

DATA_DIR = REPO_ROOT / "data"

# Raw AHN DTM strips (0.5 m, EPSG:28992)
RAW_TIF = {
    "amsterdam": DATA_DIR / "amsterdam_centre_strip.tif",
    "eindhoven": DATA_DIR / "eindhoven_north_strip.tif",
}

# Fully imputed rasters saved by TDA_Data_Imputation.ipynb
IMPUTED_NPY = {
    "amsterdam": REPO_ROOT / "output"             / "ams_final_imputed.npy",
    "eindhoven": REPO_ROOT / "imputed_datasets"   / "ehv_final_imputed.npy",
}

# Verify all files exist before proceeding
for label, path in {**RAW_TIF, **IMPUTED_NPY}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}]  {path.name}")

# ---------------------------------------------------------------------------
# Zoom AOIs: same 1 x 1 km windows used in 01_basin_persistence.ipynb.
# Coordinates are in RD New (EPSG:28992): (x_min, y_min, x_max, y_max).
# ---------------------------------------------------------------------------
ZOOM_BBOX = {
    "amsterdam": (121_500, 486_500, 122_500, 487_500),  # canal ring / Jordaan
    "eindhoven": (161_500, 383_500, 162_500, 384_500),  # Dommel valley
}

# Water-surface elevations used to fill NoData pixels before TDA.
# Raw DTM has nodata over open water; we pin those pixels to the known
# local water level so the filtration is physically correct.
# Imputed arrays already have those pixels filled — but we keep this
# constant for the raw pipeline and documentation.
WATER_LEVEL = {
    "amsterdam": -0.40,   # Amsterdam stadsboezem (regulated canal ring), m NAP
    "eindhoven": +14.0,   # Dommel river surface in the zoom AOI, m NAP
}

# Output folder for .npz persistence results
OUT_DIR = REPO_ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

print(f"\nREPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR  = {DATA_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")

## Data Extraction — Zoom Windows

We work exclusively on **1 × 1 km zoom AOIs** (2000 × 2000 pixels at native 0.5 m resolution, ~32 MB each). This avoids any memory issue: we never load the full 250 M-pixel rasters into RAM.

Two loading strategies depending on the file format:

- **Raw `.tif`** → rasterio **windowed read**: only the pixels inside the bounding box are read from disk; the rest of the file is never touched.
- **Imputed `.npy`** → NumPy **memory map** (`mmap_mode='r'`): the file is mapped into the virtual address space but pages are only fetched when actually accessed. We take the slice we need, `.copy()` it into a real array (~32 MB), then release the map.

After extraction all four crops sit in memory simultaneously at a combined ~128 MB — well within budget.

**On NaN values in the zoom crops:**

| Version | NaN meaning |
|---|---|
| Raw | Open water (canals, river) + occasional rooftop / vegetation holes |
| Imputed | Ideally zero — IDW has filled everything. Any residual NaN is flagged. |

Before running TDA, raw crops have their remaining NaN filled with the known local water-surface elevation (Amsterdam stadsboezem: −0.40 m NAP; Dommel: +14.0 m NAP). This is the physically correct treatment: open water is flat at the regulated level, so the filtration rises through it like a real inundation.

In [ ]:
# Cell 3 — loader helpers
#
# load_tif_window  : windowed read from a .tif — never loads the full raster.
# load_npy_window  : mmap slice from a .npy — uses the matching .tif only for
#                    its transform/CRS metadata, not for pixel data.

NODATA_MAGNITUDE = 1e30  # sentinel: any |value| larger than this → NaN


def _mask_nodata(arr, nodata):
    arr = arr.astype(np.float32, copy=True)
    if nodata is not None and np.isfinite(nodata):
        arr[arr == np.float32(nodata)] = np.nan
    arr[np.abs(arr) > NODATA_MAGNITUDE] = np.nan
    return arr


def load_tif_window(tif_path, bbox):
    """Read only the pixels inside bbox from a GeoTIFF.

    Parameters
    ----------
    tif_path : Path
        Path to the .tif file.
    bbox : tuple
        (x_min, y_min, x_max, y_max) in the file's CRS (RD New, EPSG:28992).

    Returns
    -------
    arr : np.ndarray float32   — 2-D crop, NaN over nodata
    transform : rasterio.Affine
    crs : rasterio.CRS
    """
    xmin, ymin, xmax, ymax = bbox
    with rasterio.open(tif_path) as src:
        win = window_from_bounds(xmin, ymin, xmax, ymax, src.transform)
        arr = src.read(1, window=win)
        transform = src.window_transform(win)
        crs = src.crs
        nodata = src.nodata
    return _mask_nodata(arr, nodata), transform, crs


def load_npy_window(npy_path, tif_path, bbox):
    """Slice the zoom window from a full-raster .npy via memory map.

    The matching .tif is opened *only* to read the transform (no pixel I/O).
    The .npy is never fully loaded into RAM.

    Parameters
    ----------
    npy_path : Path
        Path to the .npy file (full raster, same grid as the .tif).
    tif_path : Path
        Corresponding raw .tif — used for transform/CRS metadata only.
    bbox : tuple
        (x_min, y_min, x_max, y_max) in EPSG:28992.

    Returns
    -------
    arr : np.ndarray float32   — 2-D crop, NaN over nodata (if any residual)
    transform : rasterio.Affine
    crs : rasterio.CRS
    """
    xmin, ymin, xmax, ymax = bbox
    with rasterio.open(tif_path) as src:
        win = window_from_bounds(xmin, ymin, xmax, ymax, src.transform)
        row_off = int(win.row_off)
        col_off = int(win.col_off)
        height  = int(win.height)
        width   = int(win.width)
        transform = src.window_transform(win)
        crs = src.crs
        nodata = src.nodata

    arr_mmap = np.load(npy_path, mmap_mode="r")            # virtual mapping only
    crop = arr_mmap[row_off:row_off + height,
                    col_off:col_off + width].copy()         # ~32 MB real copy
    del arr_mmap                                            # release mapping

    return _mask_nodata(crop, nodata), transform, crs

In [ ]:
# Cell 4 — load all four zoom crops and report stats
#
# After this cell, `crops` holds four ~32 MB float32 arrays.
# Total RAM used: ~128 MB — no memory pressure.

crops = {}

for city in ("amsterdam", "eindhoven"):
    bbox = ZOOM_BBOX[city]

    raw_arr, tr, crs = load_tif_window(RAW_TIF[city], bbox)
    imp_arr, _,  _   = load_npy_window(IMPUTED_NPY[city], RAW_TIF[city], bbox)

    crops[city] = {
        "raw":       raw_arr,
        "imputed":   imp_arr,
        "transform": tr,
        "crs":       crs,
    }

    nan_raw = np.isnan(raw_arr).mean() * 100
    nan_imp = np.isnan(imp_arr).mean() * 100

    print(f"{city.capitalize()}:")
    print(f"  shape          {raw_arr.shape}  ({raw_arr.nbytes / 1e6:.1f} MB each)")
    print(f"  raw   NaN      {nan_raw:.2f}%")
    print(f"  imputed NaN    {nan_imp:.2f}%  {'✓ fully filled' if nan_imp == 0 else '⚠ residual NaN present'}")
    print(f"  raw   elev     [{np.nanmin(raw_arr):.2f}, {np.nanmax(raw_arr):.2f}] m NAP")
    print(f"  imputed elev   [{np.nanmin(imp_arr):.2f}, {np.nanmax(imp_arr):.2f}] m NAP")
    print()

In [ ]:
# Cell 5 — 2 × 2 heatmap: city (row) × version (col)
#
# Shared colour scale per city so raw and imputed are directly comparable.
# NaN pixels appear white (raw open water / data gaps are immediately visible).

fig, axes = plt.subplots(2, 2, figsize=(14, 12), facecolor="white")
fig.suptitle("Zoom AOI — Raw vs Imputed Elevation\n(1 × 1 km at 0.5 m resolution)",
             fontsize=14, fontweight="bold", y=1.01)

city_labels = {"amsterdam": "Amsterdam — canal ring / Jordaan",
               "eindhoven": "Eindhoven — Dommel valley"}
col_labels  = ["Raw AHN DTM", "Imputed"]

for row, city in enumerate(("amsterdam", "eindhoven")):
    arr_raw = crops[city]["raw"]
    arr_imp = crops[city]["imputed"]

    # Shared colour range from the imputed array (NaN-free, full range)
    vmin = float(np.nanmin(arr_imp))
    vmax = float(np.nanmax(arr_imp))

    for col, (arr, col_label) in enumerate([(arr_raw, "Raw AHN DTM"),
                                             (arr_imp, "Imputed")]):
        ax = axes[row, col]
        cmap = plt.get_cmap("gist_earth").copy()
        cmap.set_bad("white")          # NaN → white

        im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax,
                       interpolation="nearest")
        ax.set_title(f"{city_labels[city]}\n{col_label}", fontsize=11)
        ax.axis("off")

        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.85)
        cb.set_label("Elevation (m NAP)", fontsize=9)

        # Annotate NaN percentage
        nan_pct = np.isnan(arr).mean() * 100
        ax.text(0.02, 0.97, f"NoData: {nan_pct:.2f}%",
                transform=ax.transAxes, va="top", ha="left", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.75", alpha=0.9))

plt.tight_layout()
plt.savefig(OUT_DIR / "step2_zoom_heatmaps.png", dpi=150,
            facecolor="white", bbox_inches="tight")
print(f"Saved → {OUT_DIR / 'step2_zoom_heatmaps.png'}")
plt.show()

## Sublevel-Set Persistent Homology

We build a **2D cubical complex** from each crop using `gudhi.CubicalComplex(top_dimensional_cells=elevation)`. Gudhi's V-construction assigns to each cell the minimum of its adjacent top-dimensional cells, so the induced filtration is exactly $\{x : f(x) \le t\}$ — raising $t$ is raising the water level.

### NaN treatment before TDA

`gudhi` requires a finite value at every cell.

| Version | Fill rule |
|---|---|
| **Raw** | NaN (open water / holes) → `WATER_LEVEL[city]` — the known regulated surface (−0.40 m for Amsterdam stadsboezem; +14.0 m for the Dommel). Physically correct: open water is flat at that level. |
| **Imputed** | Should be NaN-free after IDW. Any residual NaN (flagged in Cell 4) → same `WATER_LEVEL` as fallback. |

Note: Amsterdam's imputed canals were set to **−3.0 m NAP** during the flood-fill step (Step 3 of `TDA_Data_Imputation.ipynb`), which is deeper than the regulated −0.40 m surface. This is intentional — it captures the true bathymetric floor of the canal. We therefore expect the imputed persistence diagram to show canal basins **born deeper** (lower birth coordinate) than in the raw diagram.

### Reading the persistence diagram

Each finite $H_0$ pair $(b, d)$ sits **above the diagonal** $b = d$. Its vertical distance to the diagonal equals its **persistence** $p = d - b$ (spillover depth in metres). The single **essential** pair (global minimum — born but never dying) is plotted as a triangle at the top of the axis. Low-persistence cloud near the diagonal = noise; isolated high-persistence points far above = genuine flood-risk basins.

In [ ]:
# Cell 6 — TDA helpers

def prepare_for_tda(arr, water_level):
    """Fill NaN pixels with `water_level` and cast to float64 for gudhi.

    For raw crops: NaN = open water → pinned to the regulated surface.
    For imputed crops: NaN should be absent; water_level acts as fallback only.
    """
    filled = np.where(np.isnan(arr), water_level, arr).astype(np.float64)
    return filled


def run_persistence(arr, water_level):
    """Build a 2D cubical complex and return H0 pairs as an (N, 2) array.

    Returns
    -------
    h0 : np.ndarray shape (N, 2)
        Columns: [birth, death]. Essential pair has death = +inf.
    filled : np.ndarray float64
        The NaN-filled elevation array that was passed to gudhi.
    """
    filled = prepare_for_tda(arr, water_level)
    cc = gd.CubicalComplex(top_dimensional_cells=filled)
    cc.compute_persistence(homology_coeff_field=2, min_persistence=0.0)
    pairs = cc.persistence()
    h0 = np.array([(b, d) for dim, (b, d) in pairs if dim == 0], dtype=float)
    return h0, filled

In [ ]:
# Cell 7 — compute persistence for all four crops
#
# Raw results are loaded from the .npz files already saved by
# 01_basin_persistence.ipynb (outputs/persistence_{city}_zoom.npz).
# This avoids re-running gudhi on the raw data (~60 s per crop saved).
#
# TDA is only run fresh on the two imputed crops.

results = {}

for city in ("amsterdam", "eindhoven"):
    results[city] = {}
    wl = WATER_LEVEL[city]

    # --- Raw: load from 01_basin_persistence.ipynb output ---
    npz_path = OUT_DIR / f"persistence_{city}_zoom.npz"
    if not npz_path.exists():
        raise FileNotFoundError(
            f"{npz_path.name} not found. "
            "Run all cells in 01_basin_persistence.ipynb first to generate it."
        )
    raw_npz = np.load(npz_path)
    results[city]["raw"] = {
        "h0":     raw_npz["h0"],
        "filled": raw_npz["elevation"],
    }
    fin = np.isfinite(raw_npz["h0"][:, 1])
    pers = raw_npz["h0"][fin, 1] - raw_npz["h0"][fin, 0]
    print(f"Loaded   {city} / raw    from {npz_path.name}  "
          f"— {len(raw_npz['h0']):,} H0 pairs  "
          f"max persistence = {pers.max():.3f} m")

    # --- Imputed: run TDA now ---
    arr = crops[city]["imputed"]
    print(f"Running TDA  {city} / imputed  (shape {arr.shape}) ...", end=" ", flush=True)
    h0, filled = run_persistence(arr, wl)
    results[city]["imputed"] = {"h0": h0, "filled": filled}
    fin2 = np.isfinite(h0[:, 1])
    pers2 = h0[fin2, 1] - h0[fin2, 0]
    print(f"done — {len(h0):,} H0 pairs  "
          f"max persistence = {pers2.max():.3f} m" if len(pers2) else "done — no finite pairs")
    print()

In [ ]:
# Cell 8 — 2 × 2 persistence diagrams
#
# Layout: rows = cities, columns = raw / imputed.
# Axis limits are shared per city so the two diagrams are directly comparable.
# Each subplot shows:
#   • finite H0 pairs as blue dots (birth, death)
#   • essential H0 pair (death = +inf) as blue triangle at the top of the axis
#   • diagonal y = x in grey dashes
#   • annotation box: pair count + max finite persistence

fig, axes = plt.subplots(2, 2, figsize=(13, 12), facecolor="white")
fig.suptitle("H₀ Persistence Diagrams — Sublevel-Set Filtration\n"
             "(each point = one basin; distance to diagonal = spillover depth)",
             fontsize=13, fontweight="bold", y=1.01)

for row, city in enumerate(("amsterdam", "eindhoven")):
    # Shared axis range: derive from the filled arrays (both versions)
    all_vals = np.concatenate([
        results[city]["raw"]["filled"].ravel(),
        results[city]["imputed"]["filled"].ravel(),
    ])
    pad = 0.05 * (all_vals.max() - all_vals.min())
    lo  = all_vals.min() - pad
    hi  = all_vals.max() + pad

    for col, version in enumerate(("raw", "imputed")):
        ax   = axes[row, col]
        h0   = results[city][version]["h0"]

        finite   = np.isfinite(h0[:, 1])
        h0_fin   = h0[finite]
        h0_inf   = h0[~finite]
        pers_fin = h0_fin[:, 1] - h0_fin[:, 0] if len(h0_fin) else np.array([])

        # Diagonal
        ax.plot([lo, hi], [lo, hi], "--", color="0.6", lw=0.9, zorder=1)

        # Finite pairs
        if len(h0_fin):
            ax.scatter(h0_fin[:, 0], h0_fin[:, 1],
                       s=8, c="C0", alpha=0.55, linewidths=0, zorder=2,
                       label=f"H₀ finite ({len(h0_fin):,})")

        # Essential pair(s)
        if len(h0_inf):
            ax.scatter(h0_inf[:, 0], np.full(len(h0_inf), hi * 0.97),
                       s=60, c="C0", marker="^", zorder=3,
                       label=f"H₀ essential ({len(h0_inf)})")

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_aspect("equal")
        ax.set_xlabel("birth (m NAP)", fontsize=9)
        ax.set_ylabel("death (m NAP)", fontsize=9)
        ax.set_title(f"{city.capitalize()} — {version}", fontsize=11)
        ax.legend(loc="lower right", fontsize=8, framealpha=0.85)

        # Annotation box
        if len(pers_fin):
            txt = (f"pairs: {len(h0_fin):,}\n"
                   f"max persistence: {pers_fin.max():.2f} m\n"
                   f"median persistence: {np.median(pers_fin):.3f} m")
        else:
            txt = "no finite pairs"
        ax.text(0.03, 0.97, txt, transform=ax.transAxes,
                va="top", ha="left", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.8", alpha=0.9))

plt.tight_layout()
plt.savefig(OUT_DIR / "step3_persistence_diagrams.png", dpi=150,
            facecolor="white", bbox_inches="tight")
print(f"Saved → {OUT_DIR / 'step3_persistence_diagrams.png'}")
plt.show()

In [ ]:
# Cell 9 — save persistence results to .npz for Step 4
#
# Each file stores the H0 pairs, the NaN-filled elevation array,
# the affine transform (as a 6-element vector), and the water level used.
# Naming: persistence_{city}_{version}_comparison.npz

for city in ("amsterdam", "eindhoven"):
    tr = crops[city]["transform"]
    tr_vec = np.array([tr.a, tr.b, tr.c, tr.d, tr.e, tr.f])

    for version in ("raw", "imputed"):
        path = OUT_DIR / f"persistence_{city}_{version}_comparison.npz"
        np.savez(
            path,
            h0         = results[city][version]["h0"],
            elevation  = results[city][version]["filled"],
            transform  = tr_vec,
            water_level= np.float64(WATER_LEVEL[city]),
        )
        print(f"Saved → {path.name}")

## Comparison — Bottleneck Distances & Summary

### Bottleneck distance $d_B$

Given two persistence diagrams $\mathcal{D}_1$ and $\mathcal{D}_2$, the **bottleneck distance** is

$$d_B(\mathcal{D}_1, \mathcal{D}_2) = \inf_{\gamma} \sup_{p \in \mathcal{D}_1} \| p - \gamma(p) \|_\infty$$

where $\gamma$ ranges over all bijections between the two diagrams (unmatched points are sent to their projection on the diagonal $b = d$). In practice: $d_B$ equals the cost of the worst-matched pair in the optimal matching. A small $d_B$ means the two topological summaries are nearly identical; a large $d_B$ means at least one high-persistence feature moved significantly.

We compute four distances:

| Pair | Meaning |
|---|---|
| $d_B(\text{ams\_raw},\ \text{ams\_imputed})$ | How much Amsterdam's topology changed after imputation |
| $d_B(\text{ehv\_raw},\ \text{ehv\_imputed})$ | How much Eindhoven's topology changed after imputation |
| $d_B(\text{ams\_raw},\ \text{ehv\_raw})$ | Structural difference between cities on raw data |
| $d_B(\text{ams\_imputed},\ \text{ehv\_imputed})$ | Structural difference between cities on imputed data |

**Cross-city normalization.** Amsterdam sits at −3 to +10 m NAP; Eindhoven at +10 to +35 m NAP. A raw bottleneck comparison would be dominated by this elevation offset rather than by topological structure. For the cross-city pairs we therefore **center** each diagram by subtracting its minimum birth value, so both diagrams start at 0 m. The intra-city pairs need no centering since they share the same elevation datum.

In [ ]:
# Cell 10 — bottleneck distances
#
# gudhi.bottleneck_distance expects finite-only (birth, death) arrays.
# The essential pair (death = +inf) is excluded before the call.

def finite_diagram(h0):
    """Return only the finite H0 pairs as a float64 array."""
    mask = np.isfinite(h0[:, 1])
    return h0[mask].astype(np.float64)


def center_diagram(dgm):
    """Translate a diagram so that its minimum birth = 0.
    Preserves persistence values (death - birth) exactly."""
    if len(dgm) == 0:
        return dgm
    shift = dgm[:, 0].min()
    return dgm - shift   # subtracts shift from both birth and death


# Extract finite diagrams
dgm = {
    city: {ver: finite_diagram(results[city][ver]["h0"])
           for ver in ("raw", "imputed")}
    for city in ("amsterdam", "eindhoven")
}

# Four bottleneck distances
db_ams   = gd.bottleneck_distance(dgm["amsterdam"]["raw"],
                                   dgm["amsterdam"]["imputed"])

db_ehv   = gd.bottleneck_distance(dgm["eindhoven"]["raw"],
                                   dgm["eindhoven"]["imputed"])

db_cross_raw = gd.bottleneck_distance(center_diagram(dgm["amsterdam"]["raw"]),
                                       center_diagram(dgm["eindhoven"]["raw"]))

db_cross_imp = gd.bottleneck_distance(center_diagram(dgm["amsterdam"]["imputed"]),
                                       center_diagram(dgm["eindhoven"]["imputed"]))

print("Bottleneck distances (metres)")
print(f"  d_B(ams raw,      ams imputed)   = {db_ams:.4f} m   [imputation effect, Amsterdam]")
print(f"  d_B(ehv raw,      ehv imputed)   = {db_ehv:.4f} m   [imputation effect, Eindhoven]")
print(f"  d_B(ams raw,      ehv raw)       = {db_cross_raw:.4f} m   [cross-city, raw — centered]")
print(f"  d_B(ams imputed,  ehv imputed)   = {db_cross_imp:.4f} m   [cross-city, imputed — centered]")

In [ ]:
# Cell 11 — sorted-persistence curves
#
# For each city: two curves (raw vs imputed) on the same log-log axes.
# X = rank (1 = most persistent basin), Y = persistence in metres.
# The gap between the curves shows how imputation redistributes topological mass:
#   - curve shifts up   → imputation revealed deeper/longer-lived basins
#   - curve shifts down → imputation smoothed away features (merged basins)
#   - curves overlap    → topology largely unchanged

fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor="white")
fig.suptitle("Sorted H₀ Persistence — Raw vs Imputed\n"
             "(rank 1 = deepest basin; slope reflects how quickly features become noise)",
             fontsize=12, fontweight="bold")

styles = {"raw": dict(lw=1.5, ls="-",  color="C0"),
          "imputed": dict(lw=1.5, ls="--", color="C1")}

for ax, city in zip(axes, ("amsterdam", "eindhoven")):
    for version in ("raw", "imputed"):
        h0  = results[city][version]["h0"]
        fin = np.isfinite(h0[:, 1])
        p   = np.sort(h0[fin, 1] - h0[fin, 0])[::-1]   # descending
        if len(p) == 0:
            continue
        ax.plot(np.arange(1, len(p) + 1), p,
                label=f"{version}  (n={len(p):,})",
                **styles[version])

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("rank", fontsize=10)
    ax.set_ylabel("persistence (m) — spillover depth", fontsize=10)
    ax.set_title(city.capitalize(), fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, which="both", alpha=0.25)

plt.tight_layout()
plt.savefig(OUT_DIR / "step4_sorted_persistence.png", dpi=150,
            facecolor="white", bbox_inches="tight")
print(f"Saved → {OUT_DIR / 'step4_sorted_persistence.png'}")
plt.show()

In [ ]:
# Cell 12 — summary table

def diagram_stats(h0):
    fin  = np.isfinite(h0[:, 1])
    pers = h0[fin, 1] - h0[fin, 0]
    return dict(
        n_pairs   = len(h0),
        n_finite  = int(fin.sum()),
        n_ess     = int((~fin).sum()),
        max_pers  = float(pers.max())    if len(pers) else float("nan"),
        med_pers  = float(np.median(pers)) if len(pers) else float("nan"),
        p75_pers  = float(np.percentile(pers, 75)) if len(pers) else float("nan"),
    )

rows = []
for city in ("amsterdam", "eindhoven"):
    for version in ("raw", "imputed"):
        s = diagram_stats(results[city][version]["h0"])
        rows.append((city.capitalize(), version, s))

# Text table
header = f"{'City':<12} {'Version':<10} {'H0 pairs':>9} {'Essential':>10} {'Max pers':>10} {'Median pers':>12} {'P75 pers':>10}"
sep    = "-" * len(header)
print(sep)
print(header)
print(sep)
for city, ver, s in rows:
    print(f"{city:<12} {ver:<10} {s['n_pairs']:>9,} {s['n_ess']:>10} "
          f"{s['max_pers']:>10.3f} {s['med_pers']:>12.4f} {s['p75_pers']:>10.4f}")
print(sep)

print()
print("Bottleneck distances")
print(f"  d_B(ams raw ↔ ams imputed)            = {db_ams:.4f} m")
print(f"  d_B(ehv raw ↔ ehv imputed)            = {db_ehv:.4f} m")
print(f"  d_B(ams raw ↔ ehv raw)     [centered] = {db_cross_raw:.4f} m")
print(f"  d_B(ams imp ↔ ehv imp)     [centered] = {db_cross_imp:.4f} m")